# Question 9 - 55. Jump Game

You are given an integer array `nums`. You are initially positioned at the array's **first index**, and each element in the array represents your **maximum** jump length at that position.

Return `true` if you can reach the last index, or `false` otherwise.

**Example 1:**

    Input: nums = [2,3,1,1,4]
    Output: true
    Explanation: Jump 1 step from index 0 to 1, then 3 steps to the last index.

**Example 2:**

    Input: nums = [3,2,1,0,4]
    Output: false
    Explanation: You will always arrive at index 3 no matter what. Its maximum jump length is 0, which makes it impossible to reach the last index.

**Constraints:**

- `1 <= nums.length <= 10^4`
- `0 <= nums[i] <= 10^5`

---
## 🎯 Interview Walkthrough: Brute Force → Optimized

**Pattern:** Greedy (track the farthest reachable index)

| Approach | Time | Space |
|---|---|---|
| 1️⃣ Brute force | O(2ⁿ) | O(n) |
| 1½ Memoized DP | O(n²) | O(n) |
| 2️⃣ Optimized | O(n) | O(1) |

> 🗣️ lines are an **interview script**: roughly what to say out loud at each step.

### Step 0 · Clarify before coding

🗣️ *"Let me restate it: each value is the **max** jump length, so from index `i` I can land on any of `i+1 … i+nums[i]`. I only need a yes/no answer, not the number of jumps or the path."*

- `nums[i]` is a **maximum**, so shorter jumps are allowed? → **Yes.** This matters a lot: it means reachable indices always form one solid block with no gaps.
- Values can be `0`? → **Yes.** A `0` is a trap: once you land there, you can't move.
- Single element? → Already at the last index → `true`.
- Can I modify the input? → I won't need to.

### Step 1 · Brute force

**Idea:** Try every jump length from every position. `canReach(i)` is true if `i` is the last index, or if any `canReach(i + step)` is true for `step` in `1..nums[i]`.

🗣️ *"The most direct approach is to try every path recursively. From each index I branch into up to `nums[i]` choices. That's exponential in the worst case, because the same index gets re-explored from many different starting points."*

**Why it's slow:** in `[1,1,1,…,1,0,…]`-style or large-value inputs, index `k` is reached by many different paths, and each one re-solves `canReach(k)` from scratch.

In [1]:
class SolutionBrute:
    def canJump(self, nums: list[int]) -> bool:
        last = len(nums) - 1

        def can_reach(i: int) -> bool:
            if i >= last:
                return True
            for step in range(1, nums[i] + 1):
                if can_reach(i + step):
                    return True
            return False

        return can_reach(0)

#### Step 1½ · First improvement: memoize (DP)

The fix for "re-solving the same index" is a cache: `good[i]` = "from `i` can I reach the end?". Fill it **right to left**, so every answer I look up is already computed.

🗣️ *"Each index's answer depends only on indices to its right, so I can do bottom-up DP from the end. That's O(n) states, but each state may check up to `nums[i]` neighbours, so worst case is O(n²). With n = 10⁴ that's about 10⁸ steps, which is borderline. Let me look for something linear."*

In [2]:
class SolutionDP:
    def canJump(self, nums: list[int]) -> bool:
        n = len(nums)
        good = [False] * n
        good[-1] = True
        for i in range(n - 2, -1, -1):
            farthest = min(i + nums[i], n - 1)
            good[i] = any(good[j] for j in range(i + 1, farthest + 1))
        return good[0]

### Step 2 · Optimize

**Bottleneck:** The DP asks "which of my neighbours are good?" for every index. That inner loop is the O(n) factor we want to remove.

**Key insight:** Because a jump value is a *maximum*, the set of indices I can reach is always **one solid prefix `[0 … farthest]`**. If I can reach index 7, I can also reach 1–6. So I don't need to track *which* indices are reachable, only **how far** the reachable block goes. That is a single integer.

Scan left to right:
- If `i > farthest`, there's a gap I can't cross, so return `False`.
- Otherwise `i` is reachable, so extend the block: `farthest = max(farthest, i + nums[i])`.
- As soon as `farthest >= last`, return `True`.

🗣️ *"Since the reachable positions are always a contiguous prefix, I only need to remember its right edge. I walk the array once. If I ever step past the edge I'm stuck, and otherwise I keep pushing the edge out. That's O(n) time and O(1) space."*

#### Choosing the data structure & algorithm: why this, why not that

| Option | Verdict | Why / why not |
|---|---|---|
| Recursion / DFS over all paths | ❌ | Correct, but exponential. It revisits the same index again and again. |
| BFS over indices + `visited` set | ⚠️ | Visits each index once, but enqueueing all neighbours is still O(n²) edges in the worst case. It also needs an O(n) queue and set. |
| Memoized / bottom-up DP array | ⚠️ | Removes repeated work, but the inner "any good neighbour?" loop keeps it O(n²) and it uses O(n) space. |
| **Greedy: one `farthest` integer** | ✅ | Uses the contiguity property. One pass, constant memory, and no data structure needed at all. |
| Greedy backwards (`goal` pointer moving left) | ✅ | Equally good: move `goal` to `i` whenever `i + nums[i] >= goal`, then check `goal == 0`. It's a fine alternative to mention. |

**Why is greedy safe here?** We never *commit* to a jump. `farthest` summarises *all* paths at once, so we can't make a wrong choice that we'd need to undo.

In [3]:
class Solution:
    def canJump(self, nums: list[int]) -> bool:
        farthest = 0
        last = len(nums) - 1
        for i, jump in enumerate(nums):
            if i > farthest:          # gap: index i is unreachable
                return False
            farthest = max(farthest, i + jump)
            if farthest >= last:      # early exit
                return True
        return True

### Step 3 · Toy example walkthrough

**Example A:** `nums = [2, 3, 1, 1, 4]`, last = 4

| i | nums[i] | i > farthest? | i + nums[i] | farthest after | note |
|---|---|---|---|---|---|
| 0 | 2 | 0 > 0 ❌ | 2 | 2 | can reach 0..2 |
| 1 | 3 | 1 > 2 ❌ | 4 | **4** | 4 ≥ last → `True` ✅ |

We stop at i = 1 without looking at the rest.

**Example B:** `nums = [3, 2, 1, 0, 4]`, last = 4

| i | nums[i] | i > farthest? | i + nums[i] | farthest after |
|---|---|---|---|---|
| 0 | 3 | ❌ | 3 | 3 |
| 1 | 2 | ❌ | 3 | 3 |
| 2 | 1 | ❌ | 3 | 3 |
| 3 | 0 | ❌ | 3 | 3 |
| 4 | 4 | **4 > 3 ✅ gap** | n/a | `False` |

Every route ends on the `0` at index 3. The `farthest` value shows this directly: it gets stuck at 3.

In [4]:
cases = [
    ([2, 3, 1, 1, 4], True),
    ([3, 2, 1, 0, 4], False),
    ([0], True),               # already at the end
    ([0, 1], False),           # stuck at start
    ([1, 0, 1, 0], False),
    ([2, 0, 0], True),
    ([1, 1, 1, 1], True),
    ([5, 0, 0, 0, 0, 0], True),
]
for nums, expected in cases:
    assert SolutionBrute().canJump(nums) == expected
    assert SolutionDP().canJump(nums) == expected
    assert Solution().canJump(nums) == expected
    print(nums, "->", expected)

# randomized cross-check against brute force
import random
for _ in range(500):
    nums = [random.randint(0, 3) for _ in range(random.randint(1, 10))]
    assert Solution().canJump(nums) == SolutionBrute().canJump(nums), nums
print("All tests passed ✅")

[2, 3, 1, 1, 4] -> True
[3, 2, 1, 0, 4] -> False
[0] -> True
[0, 1] -> False
[1, 0, 1, 0] -> False
[2, 0, 0] -> True
[1, 1, 1, 1] -> True
[5, 0, 0, 0, 0, 0] -> True
All tests passed ✅


### Step 4 · Wrap up

| | Time | Space | Why |
|---|---|---|---|
| Brute force (recursion) | O(2ⁿ) worst | O(n) stack | Repeated subproblems |
| DP (right to left) | O(n²) | O(n) | Each index scans up to `nums[i]` neighbours |
| **Greedy `farthest`** | **O(n)** | **O(1)** | Reachable set is a prefix, so one integer describes it |

**Complexity, explained:** each index is visited at most once, and each visit does O(1) work (one compare and one `max`). So the total is O(n). We store only `farthest` and `last`, so the space is O(1).

**Edge cases:** single element (`True`), a leading `0` with n > 1 (`False`), zeros you can jump *over* (`[2,0,0]` → `True`).

**Follow-ups:**
- *Minimum number of jumps?* → LC 45 Jump Game II (next problem). It's the same "farthest" idea, used as BFS levels.
- *Return the path?* → Keep a parent pointer for each index when it first becomes reachable.

🗣️ **30-second recap:** *"Brute force tries every jump, which is exponential. Memoizing gives O(n²). The key observation is that reachable indices are always a contiguous prefix, because jumps are maximums. So I just track the farthest reachable index in one pass. If I ever land beyond it, I'm stuck. O(n) time, O(1) space."*

---
## 📎 Appendix: original reference solution

*Kept from the source repo for comparison.*

In [5]:
class Solution:
    # Greedy Algorithm: at each step, jump as far as possible
    # Time: O(n), Space: O(1)
    def canJump(self, nums: list[int]) -> bool:
        max_reach = 0
        for i in range(len(nums)):
            # if we can't reach this index, we are stuck
            if i > max_reach:
                return False
            # update max_reach with the current max_reach and the farthest we can jump from this index (i + num[i])
            max_reach = max(max_reach, i + nums[i])
        # check if max_reach is at least the last index of the array
        return max_reach >= len(nums) - 1

In [6]:
# Test cases
test_case1 = [2, 3, 1, 1, 4]
test_case2 = [3, 2, 1, 0, 4]

# Test
canJump = Solution().canJump

print(canJump(test_case1))
print(canJump(test_case2))

True
False
